In [0]:
%sql

CREATE OR REPLACE TABLE traffic_catalog.gold.gold_traffic
USING DELTA
AS

SELECT
    road_id,
    road_name,
    road_code,

    DATE_TRUNC('HOUR', event_timestamp) AS measurement_hour,

    ROUND(AVG(current_speed), 2) AS avg_current_speed,

    ROUND(AVG(free_flow_speed), 2) AS avg_free_flow_speed,

    ROUND(
        (
            1 - AVG(current_speed) / NULLIF(AVG(free_flow_speed), 0)
        ) * 100,
        2
    ) AS speed_reduction_percentage,

    ROUND(AVG(current_travel_time), 2) AS avg_travel_time,

    ROUND(AVG(free_flow_travel_time), 2) AS avg_free_flow_travel_time,

    ROUND(AVG(confidence), 3) AS avg_confidence,

    SUM(
        CASE
            WHEN road_closure = true THEN 1
            ELSE 0
        END
    ) AS closure_events,

    COUNT(*) AS measurement_count

FROM traffic_catalog.silver.silver_traffic

GROUP BY
    road_id,
    road_name,
    road_code,
    DATE_TRUNC('HOUR', event_timestamp);

In [0]:
%sql
select * from
traffic_catalog.gold.gold_traffic
limit 5